# 02 — Latitude–depth EOFs, predictor selection and AMOC reconstruction

Reproduces the main analyses of *Low-dimensional hydrographic predictors of AMOC variability in climate models*
from the compact files written by `01_extract_and_save_data.ipynb` (no server access needed).

Pipeline (same as `Paper_notebook.ipynb`, rewritten in numpy):
1. **EOFs** for each section (variable × longitude). The weights are √(cos φ · Δz). EOFs are fitted on the training period (≤ 1989) with all members stacked. All years are then projected onto them to give member PCs, and the ensemble-mean PC is the mean of the member PCs.
2. **Ranking** of candidate predictors (variable, longitude, mode, lag 3–20 yr) by their |corr| with the AMOC over the training period.
3. **Exhaustive search** for the best k = 1, 2, 3 subsets among the top `N_CANDIDATES` features (ridge α = 1, training R²).
4. **Out-of-sample evaluation** on 1990–2014 (Table 1 / Extended Data Table 1).

Key figures: Fig. 1 (EOF + PC of one model), Fig. 4 (reconstructions), Fig. 5 (early-warning indicators).

In [ ]:
import os, itertools, warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================
DATA_DIR, RES_DIR = "./data", "./results"
os.makedirs(RES_DIR, exist_ok=True)

MODELS   = ["CESM2", "EC-Earth3", "IPSL-CM6A-LR", "MPI-ESM1-2-LR"]
VARS     = ["thetao", "so"]
LON_TAGS = ["W10p0", "W20p0", "W30p0", "W40p0", "W60p0"]
N_MODES  = 10
TRAIN_END_YEAR = 1989      # train 1850-1989, test 1990-2014

AMOC_LAT = "45N"           # "45N" (main text) or "26N" (Supplementary)
MEMBER   = None            # None = ensemble mean (paper) ; int = single member (Supp. Info)

# Detrending (linear fit on the training period only, extrapolated to the test period)
DETREND_FIELDS = False     # detrend T/S at each grid point before the EOF (paper text). The repo code does not -> False
DETREND_SERIES = True      # detrend PCs and AMOC before the regression ("Detrended" runs of the repo)

# Predictor selection
MIN_LAG, MAX_LAG = 3, 20   # years (PC leads AMOC)
N_CANDIDATES = 10          # exhaustive search among the top-N ranked features
K_MAX = 3
ALPHA = 1.0                # ridge, fixed (no tuning)

# True -> skip the search and use the predictor sets of Extended Data Table 1
USE_PAPER_FEATURES = False
# (var, lon, EOF mode [1-based], lag)
PAPER_SETS = {
    "CESM2": {1: [("thetao", "W30p0", 5, 7)],
              2: [("thetao", "W30p0", 2, 3), ("thetao", "W30p0", 5, 7)],
              3: [("thetao", "W10p0", 2, 4), ("thetao", "W30p0", 2, 3), ("thetao", "W30p0", 5, 7)]},
    "EC-Earth3": {1: [("thetao", "W40p0", 5, 20)],
                  2: [("thetao", "W40p0", 5, 20), ("thetao", "W20p0", 3, 15)],
                  3: [("thetao", "W40p0", 5, 20), ("thetao", "W10p0", 6, 9), ("thetao", "W20p0", 3, 15)]},
    "IPSL-CM6A-LR": {1: [("so", "W60p0", 1, 8)],
                     2: [("so", "W60p0", 1, 8), ("thetao", "W10p0", 2, 3)],
                     3: [("so", "W60p0", 1, 8), ("so", "W30p0", 1, 5), ("thetao", "W10p0", 2, 3)]},
    "MPI-ESM1-2-LR": {1: [("so", "W10p0", 2, 15)],
                      2: [("thetao", "W40p0", 6, 3), ("so", "W10p0", 2, 10)],
                      3: [("thetao", "W40p0", 6, 3), ("thetao", "W10p0", 4, 11), ("so", "W10p0", 2, 10)]},
}

FIG_MODEL  = "CESM2"       # model shown in the EOF/PC figure (Fig. 1)
EWS_WINDOW = 30            # years, rolling AR1 / variance

## 1. Section EOFs (computed once, then cached in `results/`)

In [ ]:
def detrend_train(x, years, train):
    """Remove a linear trend fitted on the training years (1D, NaN-aware)."""
    x = np.asarray(x, float)
    m = train & np.isfinite(x)
    if m.sum() < 2:
        return x
    a, b = np.polyfit(years[m], x[m], 1)
    return x - (a * years + b)

def detrend_fields(A, years, train):
    """Same, vectorised for A (member, time, space)."""
    t = years.astype(np.float32)
    tc = t[train] - t[train].mean()
    At = A[:, train]
    mean = At.mean(axis=1)
    slope = np.einsum("t,mts->ms", tc, At - mean[:, None]) / (tc ** 2).sum()
    return A - (mean[:, None] + slope[:, None] * (t - t[train].mean())[None, :, None])

def compute_eof(X, years, lat, dz, train, n_modes=N_MODES):
    """X (member, year, lat, depth) -> EOF maps (mode, lat, depth), PCs (member, year, mode), variance fraction.
    EOFs from the stacked training samples of all members, weights sqrt(cos(lat)*dz), then projection of all years."""
    nm, nt, nl, nz = X.shape
    w = np.sqrt(np.clip(np.cos(np.deg2rad(lat)), 0, None)[:, None] * dz[None, :]).ravel()
    Xf = X.reshape(nm, nt, nl * nz)
    ok = np.isfinite(Xf).all(axis=(0, 1)) & (w > 0)
    A = Xf[:, :, ok].astype(np.float32)
    if DETREND_FIELDS:
        A = detrend_fields(A, years, train)
    A *= w[ok].astype(np.float32)
    A -= A[:, train].mean(axis=(0, 1))
    Atr = A[:, train].reshape(-1, A.shape[-1])
    # SVD via the (samples x samples) Gram matrix: fast when n_samples << n_gridpoints
    lam, U = np.linalg.eigh((Atr @ Atr.T).astype(np.float64))
    order = np.argsort(lam)[::-1]
    vf = lam[order][:n_modes] / lam.sum()
    lam, U = lam[order][:n_modes], U[:, order][:, :n_modes]
    E = (Atr.T @ U.astype(np.float32)) / np.sqrt(lam).astype(np.float32)   # (space, mode), unit norm
    E *= np.sign(E[np.abs(E).argmax(axis=0), np.arange(n_modes)])         # deterministic sign
    pcs = A @ E                                                           # (member, year, mode)
    maps = np.full((n_modes, nl * nz), np.nan, np.float32)
    maps[:, ok] = E.T
    return maps.reshape(n_modes, nl, nz), pcs, vf

def get_eofs(model):
    """EOFs/PCs for all (var, lon) sections of a model; cached in results/."""
    cache = os.path.join(RES_DIR, f"{model}_eofs_train{TRAIN_END_YEAR}_dtf{int(DETREND_FIELDS)}.npz")
    if os.path.exists(cache):
        z = np.load(cache, allow_pickle=True)
        return {k: z[k] for k in z.files}
    out = {}
    for var in VARS:
        with xr.open_dataset(os.path.join(DATA_DIR, f"{model}_{var}_sections.nc")) as ds:
            years = ds["year"].values.astype(int)
            out["years"], out["depth"], dz = years, ds["depth"].values, ds["dz"].values
            for lt in LON_TAGS:
                X = ds[f"{var}_{lt}"].values
                lat = ds[f"lat_{lt}"].values
                maps, pcs, vf = compute_eof(X, years, lat, dz, years <= TRAIN_END_YEAR)
                out.update({f"{var}_{lt}_eof": maps, f"{var}_{lt}_pc": pcs,
                            f"{var}_{lt}_vf": vf, f"{var}_{lt}_lat": lat})
                print(f"  {model} {var} {lt}: {X.shape[0]} members, VF(1-3) = {np.round(vf[:3], 2)}")
    np.savez(cache, **out)
    return out

def load_amoc(model):
    """AMOC index (ensemble mean or one member) -> years, series."""
    with xr.open_dataset(os.path.join(DATA_DIR, f"{model}_amoc.nc")) as ds:
        def clean(da):
            # drop leftover singleton dims (e.g. 'x' from curvilinear msftyz grids)
            extra = [d for d in da.dims if d not in ("member", "year")]
            big = [d for d in extra if da.sizes[d] > 1]
            if big:
                raise ValueError(f"{model}: unexpected non-singleton dims {big} in {da.name} {dict(da.sizes)}")
            return da.squeeze(extra, drop=True)

        name = f"AMOC_{AMOC_LAT}_member"
        if name in ds:
            y = clean(ds[name]).transpose("member", "year").values
            y = np.nanmean(y, axis=0) if MEMBER is None else y[MEMBER]
        else:
            y = clean(ds[f"AMOC_{AMOC_LAT}_ensmean"]).values
        return ds["year"].values.astype(int), y.astype(float)

EOFS = {}
for model in MODELS:
    EOFS[model] = get_eofs(model)

## 2. Predictor selection and reconstruction

In [ ]:
def lagged(x, lag):
    out = np.full(len(x), np.nan)
    out[lag:] = x[:len(x) - lag]
    return out

def prepare(model):
    """Aligned AMOC target and a function returning any lagged (detrended) PC feature."""
    E = EOFS[model]
    ay, amoc = load_amoc(model)
    years = np.intersect1d(E["years"], ay)
    ie = np.searchsorted(E["years"], years)
    pcs = {(v, lt): (E[f"{v}_{lt}_pc"].mean(0) if MEMBER is None else E[f"{v}_{lt}_pc"][MEMBER])[ie]
           for v in VARS for lt in LON_TAGS}
    win = np.arange(MIN_LAG, len(years))               # evaluation window starts at MIN_LAG
    yrs = years[win]
    train = yrs <= TRAIN_END_YEAR
    Y = amoc[np.searchsorted(ay, years)][win]
    if DETREND_SERIES:
        Y = detrend_train(Y, yrs, train)

    def feature(var, lt, mode, lag):
        x = lagged(pcs[(var, lt)][:, mode - 1], lag)[win]
        return detrend_train(x, yrs, train) if DETREND_SERIES else x

    return dict(years=yrs, train=train, Y=Y, feature=feature, pcs=pcs, win=win)

def rank_features(P):
    rows = []
    for v, lt, m, lag in itertools.product(VARS, LON_TAGS, range(1, N_MODES + 1), range(MIN_LAG, MAX_LAG + 1)):
        x = P["feature"](v, lt, m, lag)
        ok = P["train"] & np.isfinite(x) & np.isfinite(P["Y"])
        r = np.corrcoef(x[ok], P["Y"][ok])[0, 1]
        rows.append((v, lt, m, lag, r, abs(r)))
    return (pd.DataFrame(rows, columns=["var", "lon", "mode", "lag", "corr", "abs_corr"])
              .sort_values("abs_corr", ascending=False).reset_index(drop=True))

def fit_eval(P, feats):
    X = np.column_stack([P["feature"](*f) for f in feats])
    Y = P["Y"]
    ok = np.isfinite(X).all(axis=1) & np.isfinite(Y)
    tr, te = ok & P["train"], ok & ~P["train"]
    mdl = make_pipeline(StandardScaler(), Ridge(alpha=ALPHA)).fit(X[tr], Y[tr])
    pred = np.full(len(Y), np.nan)
    pred[ok] = mdl.predict(X[ok])
    r = lambda m: np.corrcoef(Y[m], pred[m])[0, 1]
    return dict(feats=list(feats), pred=pred, r2_train=r2_score(Y[tr], pred[tr]), r_train=r(tr),
                r2_test=r2_score(Y[te], pred[te]), r_test=r(te))

def best_subsets(P, ranking):
    cands = [tuple(x) for x in ranking[["var", "lon", "mode", "lag"]].head(N_CANDIDATES).itertuples(index=False)]
    best = {}
    for k in range(1, K_MAX + 1):
        fits = [fit_eval(P, c) for c in itertools.combinations(cands, k)]
        best[k] = max(fits, key=lambda d: d["r2_train"])     # selected on TRAIN R²
    return best

def fmt(f):
    v, lt, m, lag = f
    return f"{'T' if v == 'thetao' else 'S'}({lt[1:3]}°W)-EOF{m}-lag{lag}"

RES, rows = {}, []
for model in MODELS:
    P = prepare(model)
    ranking = rank_features(P)
    fits = ({k: fit_eval(P, PAPER_SETS[model][k]) for k in range(1, K_MAX + 1)}
            if USE_PAPER_FEATURES else best_subsets(P, ranking))
    RES[model] = dict(P=P, ranking=ranking, fits=fits)
    for k, f in fits.items():
        rows.append(dict(model=model, k=k, predictors="; ".join(fmt(x) for x in f["feats"]),
                         train_R2=f["r2_train"], train_r=f["r_train"], test_R2=f["r2_test"], test_r=f["r_test"]))

table = pd.DataFrame(rows)
table.to_csv(os.path.join(RES_DIR, f"reconstruction_skill_AMOC{AMOC_LAT}.csv"), index=False)
pd.set_option("display.max_colwidth", 120)
table.round(2)   # = Extended Data Table 1 (k=3 rows = Table 1)

## 3. Key figures

In [ ]:
# --- Fig. 1 / Ext. Data Fig. 2-4 : EOF patterns + PCs of the k=3 predictors of one model
def plot_eof_pc(model, k=K_MAX):
    E, feats = EOFS[model], RES[model]["fits"][k]["feats"]
    years = E["years"]
    fig, axes = plt.subplots(len(feats), 2, figsize=(13, 3.3 * len(feats)), squeeze=False,
                             gridspec_kw=dict(width_ratios=[1.1, 1]))
    for (v, lt, m, lag), (a0, a1) in zip(feats, axes):
        name = f"{'Temperature' if v == 'thetao' else 'Salinity'} at {lt[1:3]}°W"
        eof = E[f"{v}_{lt}_eof"][m - 1]
        vmax = np.nanpercentile(np.abs(eof), 99)
        pm = a0.pcolormesh(E[f"{v}_{lt}_lat"], E["depth"], eof.T, cmap="RdBu_r", vmin=-vmax, vmax=vmax, shading="nearest")
        a0.invert_yaxis(); a0.set_ylabel("Depth (m)"); a0.set_xlabel("Latitude")
        a0.set_title(f"{name} | EOF {m} ({100 * E[f'{v}_{lt}_vf'][m - 1]:.0f} %)")
        fig.colorbar(pm, ax=a0, label="EOF amplitude")
        pc = E[f"{v}_{lt}_pc"][:, :, m - 1]
        a1.plot(years, pc.T, color="0.6", lw=0.5, alpha=0.4)
        a1.plot(years, pc.mean(0), "k", lw=1.8, label="ensemble mean")
        a1.axvline(TRAIN_END_YEAR + 0.5, color="0.4", ls="--", lw=1)
        a1.set_title(f"{name} | PC {m}  (used with lag {lag} yr)"); a1.set_ylabel("PC anomaly")
        a1.legend(loc="upper left", fontsize=8)
    fig.suptitle(f"{model}: selected EOF patterns and PCs", y=1.0)
    fig.tight_layout()
    fig.savefig(os.path.join(RES_DIR, f"fig1_eof_pc_{model}.png"), dpi=200, bbox_inches="tight")

plot_eof_pc(FIG_MODEL)   # loop over MODELS for Ext. Data Figs 2-4

In [ ]:
# --- Fig. 4 : reconstruction of the (detrended) AMOC, k = 1..3
fig, axes = plt.subplots(len(MODELS), 1, figsize=(11, 3.2 * len(MODELS)), sharex=True)
cols = {1: "tab:orange", 2: "tab:green", 3: "tab:blue"}
for ax, model in zip(np.atleast_1d(axes), MODELS):
    P, fits = RES[model]["P"], RES[model]["fits"]
    ax.plot(P["years"], P["Y"], "k", lw=1.8, label=f"AMOC {AMOC_LAT}")
    for k, f in fits.items():
        ax.plot(P["years"], f["pred"], color=cols.get(k), lw=1.2,
                label=f"k={k}  (r train {f['r_train']:.2f} / test {f['r_test']:.2f})")
    ax.axvline(TRAIN_END_YEAR + 0.5, color="0.5", ls="--", lw=1)
    ax.set_title(model); ax.set_ylabel("AMOC anomaly (Sv)"); ax.legend(fontsize=8, loc="lower left", ncol=2)
    ax.grid(alpha=0.3)
ax.set_xlabel("Year")
fig.tight_layout()
fig.savefig(os.path.join(RES_DIR, f"fig4_reconstruction_AMOC{AMOC_LAT}.png"), dpi=200, bbox_inches="tight")

In [ ]:
# --- Fig. 5 : early-warning indicators (rolling lag-1 autocorrelation and variance, 30-yr centred window)
def rolling_stats(x, window=EWS_WINDOW, min_valid=10):
    s = pd.Series(x)
    ar1 = s.rolling(window, center=True, min_periods=min_valid).apply(lambda w: pd.Series(w).autocorr(1), raw=True)
    var = s.rolling(window, center=True, min_periods=min_valid).var()
    return ar1.values, var.values

def standardize_train(x, train):
    return (x - np.nanmean(x[train])) / np.nanstd(x[train], ddof=1)

fig, axes = plt.subplots(2, len(MODELS), figsize=(4.5 * len(MODELS), 6), sharex=True)
for j, model in enumerate(MODELS):
    P = RES[model]["P"]
    series = {f"AMOC {AMOC_LAT}": P["Y"]}
    for v, lt, m, _ in RES[model]["fits"][K_MAX]["feats"]:
        series[fmt((v, lt, m, 0)).rsplit("-lag", 1)[0]] = P["feature"](v, lt, m, 0)   # unlagged PC
    for name, x in series.items():
        ar1, var = rolling_stats(standardize_train(x, P["train"]))
        style = dict(color="red", ls="--", lw=2) if name.startswith("AMOC") else dict(lw=1.3)
        axes[0, j].plot(P["years"], ar1, label=name, **style)
        axes[1, j].plot(P["years"], var, **style)
    for ax in axes[:, j]:
        ax.axvline(TRAIN_END_YEAR + 0.5, color="0.5", ls=":", lw=1); ax.grid(alpha=0.3)
    axes[0, j].set_title(model); axes[0, j].legend(fontsize=7)
axes[0, 0].set_ylabel("Lag-1 autocorrelation"); axes[1, 0].set_ylabel("Rolling variance (standardized)")
fig.tight_layout()
fig.savefig(os.path.join(RES_DIR, f"fig5_EWS_AMOC{AMOC_LAT}.png"), dpi=200, bbox_inches="tight")

## Quick variants

| Paper analysis | Setting |
|---|---|
| Exact Table 1 / Ext. Data Table 1 predictors | `USE_PAPER_FEATURES = True` |
| AMOC at 26°N (Supp. Info) | `AMOC_LAT = "26N"` |
| Single ensemble member (Supp. Info Fig. 1) | `MEMBER = 0, 1, …` (check that AMOC and T/S member orders match) |
| Field detrending before the EOF (paper text) | `DETREND_FIELDS = True` (new cache file name, no need to delete the old one) |
| Ext. Data Figs 2–4 | `for m in MODELS: plot_eof_pc(m)` |
| Lagged correlations (Ext. Data Fig. 5) | `RES[model]["ranking"]` holds corr(PC(t−lag), AMOC(t)) for all modes and lags |